## The Triage "Mess": Cleaning Data the Professional Way

### Notebook Setup

In [1]:
import pandas as pd

df = pd.read_csv("data/patient_data.csv")

# print the first five rows
print("The First Five Rows of Data: \n")
print(df.head())

print("\n", "-"* 30, "\n")
# print the data info
print("The Original Data Info: \n")
print(df.info())

The First Five Rows of Data: 

    age  body_temp symptom_type  priority
0  52.0  36.900076     Fracture  Standard
1  15.0  37.122294     Fracture  Standard
2  72.0  35.869459        Fever  Standard
3  61.0  37.404321        Other  Standard
4  21.0  36.055636     Headache  Standard

 ------------------------------ 

The Original Data Info: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           900 non-null    float64
 1   body_temp     1000 non-null   float64
 2   symptom_type  1000 non-null   object 
 3   priority      1000 non-null   object 
dtypes: float64(2), object(2)
memory usage: 31.4+ KB
None


In [2]:
df.shape

(1000, 4)

#### 1. Split the dataset

In [6]:
from sklearn.model_selection import train_test_split

# 1. Split into X (features) and y (target)
X = df[['age', 'body_temp', 'symptom_type']]
y = df['priority']

# 2. Split into Train and Test sets... ONCE. On the RAW data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"\nMissing values in X_train['age']: {X_train['age'].isna().sum()}")

X_train shape: (800, 3)
X_test shape:  (200, 3)

Missing values in X_train['age']: 76


#### 2. `fit` the Imputer

In [14]:
from sklearn.impute import SimpleImputer

# 1. Create the imputer tool
age_imputer = SimpleImputer(strategy='mean')

# 2. 'Teach' the tool by showing it the training data's 'age' column
# Fill in the ...
age_imputer.fit(X_train[['age']])

# Let's see what it learned!
print(f"The imputer learned the mean age is: {age_imputer.statistics_[0]:.1f}")

The imputer learned the mean age is: 43.0


#### 3. `transform` the Data

In [16]:
# 1. Apply the learned rule to the training set
X_train['age_imputed'] = age_imputer.transform(X_train[['age']])

# 2. Apply the *SAME* rule (43.0) to the test set
# We NEVER fit on the test set!
X_test['age_imputed'] = age_imputer.transform(X_test[['age']])

print("--- After Imputing ---")
print(f"Missing values in X_train['age_imputed']: {X_train['age_imputed'].isna().sum()}")
print(f"Missing values in X_test['age_imputed']:  {X_test['age_imputed'].isna().sum()}")

--- After Imputing ---
Missing values in X_train['age_imputed']: 0
Missing values in X_test['age_imputed']:  0


#### 4. Select Numeric Features

In [19]:
numeric_features = ['age_imputed', 'body_temp']
print(numeric_features)

['age_imputed', 'body_temp']


#### 5. `fit` and `transform` the Scaler

In [20]:
from sklearn.preprocessing import StandardScaler

# 1. Create the scaler tool
scaler = StandardScaler()

# 2. Learn the rules (mean/std) and apply them to X_train
X_train_scaled = scaler.fit_transform(X_train[numeric_features])

# 3. Apply the *SAME* rules (learned from X_train) to X_test
X_test_scaled = scaler.transform(X_test[numeric_features])

print("--- Scaled Training Data (first 5 rows) ---")
print(X_train_scaled[:5])

--- Scaled Training Data (first 5 rows) ---
[[-0.04023925 -0.87177311]
 [ 0.73393007 -0.49764839]
 [ 0.65243856  0.69920673]
 [ 0.61169281  0.16549113]
 [-1.18112034 -0.10839524]]


#### 6. `fit` and `transform` the Encoder

In [23]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['symptom_type']

# 1. Create the encoder tool
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 2. Learn the categories and apply to X_train
X_train_encoded = encoder.fit_transform(X_train[categorical_features])

# 3. Apply the *SAME* categories to X_test
X_test_encoded = encoder.transform(X_test[categorical_features])

print("--- Encoder Categories Learned ---")
print(encoder.categories_)
print("\n--- Encoded Training Data (first 5 rows) ---")
print(X_train_encoded[:5])

--- Encoder Categories Learned ---
[array(['Chest Pain', 'Fever', 'Fracture', 'Headache', 'Other'],
      dtype=object)]

--- Encoded Training Data (first 5 rows) ---
[[0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]]


#### 7. Build a Numeric `Pipeline`

In [27]:
from sklearn.pipeline import Pipeline
import numpy as np

# Re-load our original data just to start fresh
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define our numeric features list
numeric_features = ['age', 'body_temp']

# Create the numeric 'recipe'
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Now we can .fit_transform() this pipeline on our numeric columns
X_train_numeric_processed = numeric_pipeline.fit_transform(X_train[numeric_features])

# And just .transform() on the test set!
X_test_numeric_processed = numeric_pipeline.transform(X_test[numeric_features])

print("--- Processed Numeric Data (first 5 rows) ---")
print(X_train_numeric_processed[:5])
print(f"\nMissing values: {np.isnan(X_train_numeric_processed).sum()}")

--- Processed Numeric Data (first 5 rows) ---
[[-0.04023925 -0.87177311]
 [ 0.73393007 -0.49764839]
 [ 0.65243856  0.69920673]
 [ 0.61169281  0.16549113]
 [-1.18112034 -0.10839524]]

Missing values: 0


#### 8. Build the `ColumnTransformer`

In [29]:
from sklearn.compose import ColumnTransformer

# We already have the numeric pipeline from the last step

# 1. Create the categorical 'recipe' (just one step)
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 2. Define the categorical features list
categorical_features = ['symptom_type']

# 3. Assemble the 'floor manager'
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features), # (name, transformer_object, list_of_columns)
        ('cat', categorical_pipeline, categorical_features)  # (name, transformer_object, list_of_columns)
    ])

print("ColumnTransformer is ready!")

ColumnTransformer is ready!


#### 9. Use the `ColumnTransformer`

In [31]:
# 1. 'Learn' the rules from X_train and apply them
X_train_processed = preprocessor.fit_transform(X_train)

# 2. Apply the *SAME* rules to X_test
X_test_processed = preprocessor.transform(X_test)

print("--- Fully Processed Training Data ---")
print(f"Shape: {X_train_processed.shape}")
print("(2 scaled numeric columns + 5 encoded categorical columns)")
print("\n--- Fully Processed Test Data ---")
print(f"Shape: {X_test_processed.shape}")

--- Fully Processed Training Data ---
Shape: (800, 7)
(2 scaled numeric columns + 5 encoded categorical columns)

--- Fully Processed Test Data ---
Shape: (200, 7)


#### 10. Build the Final Pipeline

In [33]:
from sklearn.linear_model import LogisticRegression

# 1. Define our model
model = LogisticRegression()

# 2. Create the full pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Step 1: Our preprocessor
    ('model', model)         # Step 2: Our model
])

# Look at our beautiful, clean pipeline!
print(full_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'body_temp']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['symptom_type'])])),
                ('model', LogisticRegression())])


#### 11. Use `full_pipeline`

In [36]:
# 1. Train the *entire* pipeline on the *raw* training data
# This will auto-impute, auto-scale, auto-encode, and then train the model!
full_pipeline.fit(X_train, y_train)

# 2. Get predictions on the *raw* test data
# This will auto-transform all the test data using the *learned* rules and get a prediction
y_pred = full_pipeline.predict(X_test)

# 3. Check our score (this also does the previous step automatically!)
score = full_pipeline.score(X_test, y_test)

print(f"Our model's Test Accuracy: {score * 100:.2f}%")
print("\n--- Sample Predictions ---")
print(f"Test Data: {y_test.values[:10]}")
print(f"Predictions: {y_pred[:10]}")

Our model's Test Accuracy: 86.00%

--- Sample Predictions ---
Test Data: ['Urgent' 'Standard' 'Standard' 'Standard' 'Standard' 'Standard'
 'Standard' 'Standard' 'Urgent' 'Standard']
Predictions: ['Urgent' 'Standard' 'Standard' 'Standard' 'Standard' 'Standard'
 'Standard' 'Standard' 'Standard' 'Standard']
